## Setup

### Imports

In [ ]:
import numpy as np
import pandas as pd
import scipy as sp
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import colorcet as cc
import yaml
from statsmodels.distributions.copula.api import GumbelCopula

import gumbel_copula_2dRP as rp
import RP_plotting as rp_plot
import gumbel_copula_ESS as ess

# set up plot preferences
plt.rcParams['font.sans-serif'] = 'Helvetica'
plt.rcParams['font.size'] = 8

### Constants

In [ ]:
def load_config(config_path):
    '''Loads the configuration from a YAML file.'''
    with open(config_path, 'r') as file:
        config = yaml.safe_load(file)
    return config

config = load_config('config/config.yaml')

In [ ]:
# Config parameters
# THRESH = config['THRESH']
THRESH=20
REGIONS = config['REGIONS']
MODEL = config['MODEL']
T = config['T']
n = config['n']

# Other parameters
EXPORTS = False
cmap = cc.cm['kbc_r']
u_vals = np.linspace(0.8, 0.999, n)
v_vals = np.linspace(0.8, 0.999, n)

### Data

In [ ]:
# Drought summary data
if MODEL == 'obs':
    # Obs
    drought_data = pd.read_csv(
        '/Users/ryanvan/Library/CloudStorage/OneDrive-UniversityofVermont/Documents/_UVM/Research/CIROH/_CRB Study/Drought Data/Drought_Properties_jd.csv',
        parse_dates=['start', 'end', 'previous_end'],
        index_col='StaID'
        )
elif MODEL == 'nwm':
    # NWM
    drought_data = pd.read_csv(
        '/Users/ryanvan/Library/CloudStorage/OneDrive-UniversityofVermont/Documents/_UVM/Research/CIROH/_CRB Study/HyMED/nwm_drought_props_long.csv',
        parse_dates=['start', 'end'],
        index_col='site'
        )
else:
    # CBRFC
    drought_data = pd.read_csv(
        '/Users/ryanvan/Library/CloudStorage/OneDrive-UniversityofVermont/Documents/_UVM/Research/CIROH/_CRB Study/HyMED/cbrfc_drought_props_long.csv',
        parse_dates=['start', 'end'],
        index_col='site'
        )

# List of study gages with HCDN clusters
gages = pd.read_csv(
    '/Users/ryanvan/Library/CloudStorage/OneDrive-UniversityofVermont/Documents/_UVM/Research/CIROH/_CRB Study/CRB Gages/NWM_v3_CRB_with_HCDN_cluster.csv',
    index_col='USGS_ID'
    )[['Lat', 'Lon', 'region']]

# Regions
regions = gages['region'].unique().tolist()
region_names = [
    'Southwest',
    'California and Interior West',
    'Rocky Mountains'
]
# print(regions[REGION])

## Temporal Clustering

In [ ]:
def perform_ESS(drought_data, delta_t):
    # set up data for ESS
    ess_data = drought_data[['start', 'end', 'severity', 'duration']].copy()
    
    # Get midpoint of each event and sort chronologically
    events = ess_data.copy()
    events['event_time'] = events.apply(
        lambda r: ess.event_midpoint(r['start'], r['end']),
        axis=1
    )

    events = events.sort_values('event_time').reset_index(drop=True)
    
    # Perform temporal clustering
    clusters = ess.temporal_clustering(events, delta_t)
    n_eff = len(clusters)
    
    D_reg = []
    S_reg = []

    for cl in clusters:
        D, S = ess.aggregate_cluster(
            cl,
            duration_rule='mean',
            severity_rule='mean'
            )
        
        D_reg.append(D)
        S_reg.append(S)

    return np.asarray(D_reg), np.asarray(S_reg), n_eff
    

## Fit the Marginals and Copula Function

In [ ]:
out_duration = []   # the most likely duration values
out_severity = []   # the most likely severity values
out_upper_d = []    # CI bounds
out_lower_d = []
out_upper_s = []
out_lower_s = []
out_n_eff = []


fig, axes = plt.subplots(
    figsize=(6.5, 2.5),
    dpi=300,
    nrows=1,
    ncols=3,
    sharex=True,
    sharey=True,
    layout='constrained'
)

for ax, REGION in zip(axes, REGIONS):
    
    ax.spines[:].set_linewidth(0.5)
    ax.grid('major', linewidth=0.5, linestyle=':')
    
    # Filter to region
    data = drought_data[drought_data['threshold'] == THRESH]
    gages_in_region = gages[gages['region'] == regions[REGION]].index.tolist()
    data = data[data.index.isin(gages_in_region)]
    data = data[data != 0]
    # data = data[['duration', 'severity']]
    data.dropna(inplace=True)
    
    # Temporal sampling
    D_reg, S_reg, n_eff = perform_ESS(data, pd.Timedelta('30D'))
    out_n_eff.append(n_eff)
    
    ax.set_title(f'{region_names[REGION]} (n={n_eff})', fontsize=8)

    # Fit EVs
    duration_rv, duration_params, duration_aic = rp.best_fit_rv(D_reg, ['gamma', 'weibull_min', 'expon'], print=False)
    severity_rv, severity_params, severity_aic = rp.best_fit_rv(S_reg, ['gamma', 'weibull_min', 'expon'], print=False)

    duration_rv = duration_rv(*duration_params)
    severity_rv = severity_rv(*severity_params)
    
    # fit theta
    copula = GumbelCopula()
    theta = copula.fit_corr_param(
        np.vstack((data['duration'], data['severity'])).transpose()
        )
    
    
    all_duration = []
    all_severity = []
    all_likelihood = []
    max_duration = []       # duration value with max likelihood
    max_severity = []       # severity value with max likelihood
    CI_upper_x = [0]
    CI_upper_y = [0]
    CI_lower_x = [0]
    CI_lower_y =[0]
    
    for t in T:
    
        # compute the iso-line
        V = rp.iso_rp_OR(u_vals, t, theta)
        
        # transform to duration and severity values
        duration_val = duration_rv.ppf(u_vals)
        severity_val = severity_rv.ppf(V)
        all_duration.append(duration_val)
        all_severity.append(severity_val)

        # compute the likelihood along iso-RP
        likelihood = rp.joint_density_OR(u_vals, V, theta, duration_rv, severity_rv)
        # Normalize likelihood to [0, 1], ignoring NaN values
        min_val = np.nanmin(likelihood)
        max_val = np.nanmax(likelihood)
        if max_val != min_val:
            likelihood_normalized = (likelihood - min_val) / (max_val - min_val)
        else:
            likelihood_normalized = np.zeros_like(likelihood)
        
        all_likelihood.append(likelihood_normalized)
        
        # get the duration and severity values at the max likelihood of the iso-line
        ind_max = np.nanargmax(likelihood_normalized)
        max_duration.append(duration_val[ind_max])
        max_severity.append(severity_val[ind_max])


        
        # compute the 95% confidence interval around the max likelihood
        logL = np.log(likelihood)
        logL_max = np.nanmax(logL)
        threshold = logL_max - 0.5 * sp.stats.chi2.ppf(0.95, df=1)
        mask = logL >= threshold
        dur_ci = duration_val[mask]
        sev_ci = severity_val[mask]
        
        CI_upper_x.append(np.min(dur_ci))
        CI_upper_y.append(np.max(sev_ci))
        CI_lower_x.append(np.max(dur_ci))
        CI_lower_y.append(np.min(sev_ci))
    
    # to shade the CI area, need to interpolate to a shared x grid
    x1 = np.array(CI_upper_x)
    y1 = np.array(CI_upper_y)

    x2 = np.array(CI_lower_x)
    y2 = np.array(CI_lower_y)

    # polygon for p_fill
    x_poly = np.concatenate([
        x1,
        [x2[-1]],   # straight line to end of line 2
        x2[::-1],
        [x1[0]]     # straight line back to start of line 1
    ])

    y_poly = np.concatenate([
        y1,
        [y2[-1]],
        y2[::-1],
        [y1[0]]
    ])
    
    # plot the iso-lines colored by likelihood
    for t in range(len(T)):
        lc = rp_plot.colored_line(
            all_duration[t],
            all_severity[t],
            all_likelihood[t],
            cmap=cmap,
            norm=plt.Normalize(0, 1),
            linewidth=2)
        
        ax.add_collection(lc)


    # plot the max likelihood events
    ax.plot(
        max_duration,
        max_severity,
        linestyle='',
        marker='o',
        color='r',
        ms=4,
        label='Most likely event'
    )

    # label each max likelihood event with corresponding return period
    for i, t in enumerate(T):
        ax.text(max_duration[i] * 1.05, max_severity[i] * 1.05, f'{t} yr', fontsize=6, ha='left', va='bottom', color='r')

    # fill the CI region
    ax.fill(
        x_poly,
        y_poly,
        color='silver', #cmap(0.95),
        linewidth=0,
        alpha=0.5,
        label='95% Conf. Int.'
        )


    # proxy object for iso-line legend item
    proxy = mlines.Line2D(
        [], [],
        color=cmap(0.5),
        linewidth=2
    )
    
    out_duration.append(max_duration)
    out_severity.append(max_severity)
    out_upper_d.append(CI_upper_x[1:])
    out_lower_d.append(CI_lower_x[1:])
    out_upper_s.append(CI_upper_y[1:])
    out_lower_s.append(CI_lower_y[1:])

cbar = fig.colorbar(lc, ax=ax, ticks=[0,1], shrink=0.5, aspect=15)
ax.set_ylim(0)
ax.set_xlim(0)
fig.supxlabel('Duration (days)', fontsize=8)
fig.supylabel('Severity (percentile \u00D7 days)', fontsize=8)

cbar.ax.spines['outline'].set(visible=True, lw=0.5, edgecolor='black')
cbar.ax.set_yticklabels(['Least likely', 'Most likely'], va='center', rotation=90)

handles, labels = ax.get_legend_handles_labels()
handles.insert(0, proxy)
labels.insert(0, 'Return period')
# ax.legend(handles, labels)
fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, -0.1), ncols=3, frameon=False, fontsize=8)

## Export the data

In [ ]:
if EXPORTS:
    df = {
        'Region':       [x for x in region_names for _ in range(5)],
        'T':            T * 3,
        'Duration':     [item for sublist in out_duration for item in sublist],
        'Severity':     [item for sublist in out_severity for item in sublist],
        'D_upper':      [item for sublist in out_upper_d for item in sublist],
        'D_lower':      [item for sublist in out_lower_d for item in sublist],
        'S_upper':      [item for sublist in out_upper_s for item in sublist],
        'S_lower':      [item for sublist in out_lower_s for item in sublist]
    }

    df = pd.DataFrame(df)
    df = df.round(0)
    
    df.to_csv(f'RP_Figures/New_style/RP_data_{MODEL}_{THRESH}.csv', index=False)